# Merton jump-diffusion: adding jumps to Black-Scholes

Every model so far moves the price *continuously*: Brownian motion has continuous
paths, so over a tiny time step the price wiggles but never teleports. Merton
(1976) adds a second source of randomness, sudden discontinuous *jumps*, on top
of the diffusion. This notebook derives the Merton characteristic function from
first principles, then (Tuesday) implements and validates it, reusing the COS
and Carr-Madan Fourier machinery from Week 5.

The motivation is empirical. Last week we measured a very steep short-dated SPX
skew (rho near -0.99 at 29 DTE). A pure diffusion cannot produce that: over a
short horizon its log-return is nearly Gaussian, so its tails are thin, so deep
OTM puts should be nearly worthless and there should be almost no skew. The
market says otherwise. Jumps put genuine weight in the short-dated tail, because
a jump can occur in any interval however short, and that is exactly what a rich
deep-OTM put price requires.

## 1. The model

Under the risk-neutral measure, the log-price follows a diffusion plus a
compound Poisson jump term:

$$d(\log S_t) = \left(r - q - \tfrac12\sigma^2 - \lambda\kappa\right)dt
+ \sigma\, dW_t + dJ_t$$

Two pieces of randomness:

- **Diffusion**: $\sigma\, dW_t$, the usual Brownian part.
- **Jumps**: $dJ_t$, a compound Poisson process. Jumps arrive at Poisson rate
  $\lambda$ (average jumps per year). When a jump occurs, the log-price moves by
  a random amount $Y \sim \mathcal{N}(\mu_J, \delta_J^2)$. So Merton's jump sizes
  are **normally distributed in log-space**, which makes the jump multiplier on
  the price lognormal.

The parameters:

- $\sigma$: diffusion volatility (as in BSM).
- $\lambda$: jump intensity, expected number of jumps per year.
- $\mu_J$: mean jump size in log-space (negative $\mu_J$ = jumps tend downward,
  the crash-risk direction).
- $\delta_J$: standard deviation of jump size in log-space.

### The martingale correction

The drift term has an extra piece, $-\lambda\kappa$, that is not in BSM. It is a
compensator that keeps the discounted price a martingale (risk-neutral pricing
requires $E[S_T] = S_0 e^{(r-q)T}$). Jumps add expected growth, so the drift must
be pushed down to cancel it. The size of that expected jump contribution is

$$\kappa = E[e^Y - 1] = e^{\mu_J + \tfrac12\delta_J^2} - 1$$

which is just the expected proportional price change per jump ($e^Y$ is the price
multiplier, $E[e^Y]$ for a lognormal is $e^{\mu_J + \delta_J^2/2}$). Multiply by
$\lambda$ (jumps per year) and subtract in the drift, and the mean jump growth is
exactly compensated. This is the same martingale-correction logic as the Heston
K0 term, applied to jumps instead of stochastic variance.

## 2. The characteristic function, derived

The characteristic function of the log-price is
$\phi(u) = E[e^{iu \log S_T}]$. It is the object the COS and Carr-Madan pricers
consume, so deriving it is the whole job. The derivation rests on one clean fact:

**The diffusion and the jumps are independent.** Independent random variables
have characteristic functions that *multiply*. So the Merton char func factorises:

$$\phi_{\text{Merton}}(u) = \phi_{\text{diffusion}}(u) \cdot \phi_{\text{jump}}(u)$$

The first factor you already know, it is the BSM log-price char func (a Gaussian).
The only new work is the jump factor.

### The jump factor via conditioning on the number of jumps

Over horizon $\tau$, let $N$ be the number of jumps, which is Poisson with mean
$\lambda\tau$: $P(N = n) = e^{-\lambda\tau}(\lambda\tau)^n / n!$. Given $N = n$
jumps, the total jump contribution is a sum of $n$ iid normal jump sizes, which
is itself normal: $\sum_{i=1}^n Y_i \sim \mathcal{N}(n\mu_J, n\delta_J^2)$.

Condition on $N$ and use the tower property:

$$\phi_{\text{jump}}(u) = E\!\left[e^{iu \sum_i Y_i}\right]
= \sum_{n=0}^{\infty} P(N=n)\, E\!\left[e^{iu\sum_{i=1}^n Y_i}\right]$$

The inner expectation is the char func of a $\mathcal{N}(n\mu_J, n\delta_J^2)$,
which is $\exp(iu\, n\mu_J - \tfrac12 u^2 n\delta_J^2)$. Substitute:

$$\phi_{\text{jump}}(u)
= \sum_{n=0}^{\infty} \frac{e^{-\lambda\tau}(\lambda\tau)^n}{n!}
\exp\!\left(n\left[iu\mu_J - \tfrac12 u^2\delta_J^2\right]\right)$$

Pull the $n$-dependence together and recognise the exponential series
$\sum_n x^n/n! = e^x$:

$$\phi_{\text{jump}}(u)
= e^{-\lambda\tau}\sum_{n=0}^{\infty}
\frac{\left(\lambda\tau\, e^{iu\mu_J - \frac12 u^2\delta_J^2}\right)^n}{n!}
= \exp\!\left(\lambda\tau\left[e^{iu\mu_J - \frac12 u^2\delta_J^2} - 1\right]\right)$$

That closed form is the entire jump contribution. It is a special case of the
**Levy-Khintchine formula**: the char func of any compound Poisson process is
$\exp(\lambda\tau (\hat{f}(u) - 1))$, where $\hat{f}$ is the char func of a single
jump size. Here the single-jump char func is the Gaussian
$e^{iu\mu_J - u^2\delta_J^2/2}$.

## 3. The full Merton characteristic function

Combine the diffusion factor (BSM Gaussian log-price char func, with the
jump-compensated drift) and the jump factor:

$$\phi_{\text{Merton}}(u) = \exp\!\Big(iu\big(\log S_0 + (r - q - \tfrac12\sigma^2
- \lambda\kappa)\tau\big) - \tfrac12\sigma^2 u^2\tau\Big)
\cdot \exp\!\Big(\lambda\tau\big[e^{iu\mu_J - \frac12 u^2\delta_J^2} - 1\big]\Big)$$

with $\kappa = e^{\mu_J + \delta_J^2/2} - 1$.

Two sanity checks that fall straight out of this form, and both become tests
on Tuesday:

1. **No jumps recovers BSM.** Set $\lambda = 0$. The jump factor becomes
   $e^0 = 1$, and the drift correction $-\lambda\kappa$ vanishes, leaving exactly
   the BSM log-price char func. A jump model must reduce to Black-Scholes when
   jumps are switched off.
2. **The compensator keeps it a martingale.** At $u = -i$ the char func gives
   $E[S_T]$, which must equal $S_0 e^{(r-q)\tau}$. The $-\lambda\kappa\tau$ drift
   term is precisely what makes the jump factor cancel against it. Checking
   $\phi(-i) = S_0 e^{(r-q)\tau}$ numerically is a clean martingale test.

### Why this shape produces short-dated skew

The jump factor does not vanish as $\tau \to 0$ the way a diffusion's tail
contribution does. For any $\tau > 0$ there is probability $\approx \lambda\tau$
of at least one jump, and that jump can be large. So the short-dated return
distribution keeps a fat tail, and with $\mu_J < 0$ (downward jumps) a *left*-fat
tail specifically, which is exactly skew. The diffusion part alone would give a
nearly Gaussian, nearly symmetric, thin-tailed short-dated smile; the jump part
bends it. This is the mechanism that lets Merton reach the steep 29 DTE SPX skew
that Heston, being a pure diffusion in the price, flattens away too fast.

### The limitation, previewing Kou

Merton's jumps are *normal*, so symmetric-ish and light-tailed within the jump
distribution itself. Real crash risk is more asymmetric and heavier-tailed than
a Gaussian jump captures. Kou (Wednesday) replaces the normal jump with a
*double-exponential*, two-sided with different up and down decay rates, giving
fatter and more asymmetric tails, and analytically convenient tails that also
help with certain exotic pricing. Merton first because its Gaussian jump is the
gentler derivation and the direct generalisation of BSM.

In [ ]:
# Imports. Reuse the Week 5 Fourier machinery; jump char funcs plug straight in.
import numpy as np
import matplotlib.pyplot as plt

# from pricing.fourier import cos_price          # the COS pricer from W5
# from pricing.carr_madan import carr_madan_price  # optional cross-check
# from models.bsm import bsm_price                # for the lambda -> 0 limit test

In [ ]:
# models/merton.py will hold this. Stub here for notebook development.
#
# def merton_char_func(u, S0, r, q, T, sigma, lam, mu_j, delta_j):
#     """Merton jump-diffusion characteristic function of log S_T.
#
#     phi(u) = exp( iu(log S0 + (r - q - 0.5 sigma^2 - lam*kappa) T)
#                   - 0.5 sigma^2 u^2 T )
#              * exp( lam T ( exp(iu mu_j - 0.5 u^2 delta_j^2) - 1 ) )
#     with kappa = exp(mu_j + 0.5 delta_j^2) - 1.
#     """
#     # kappa = ...                          # martingale compensator
#     # drift = log(S0) + (r - q - 0.5*sigma**2 - lam*kappa) * T
#     # diffusion_part = exp(1j*u*drift - 0.5 * sigma**2 * u**2 * T)
#     # jump_part = exp(lam * T * (exp(1j*u*mu_j - 0.5*u**2*delta_j**2) - 1))
#     # return diffusion_part * jump_part
#     raise NotImplementedError

In [ ]:
# Validation, ground-truth-first as always.
#
# Gate 1: lambda -> 0 recovers BSM.
#   Price a call via COS using merton_char_func with lam=0, compare to bsm_price.
#   Should agree to COS truncation accuracy.
#
# Gate 2: martingale check.
#   phi(-i) must equal S0 * exp((r - q) T). Evaluate the char func at u = -1j,
#   assert close to S0*exp((r-q)*T). This certifies the kappa compensator.
#
# Gate 3: COS vs Monte Carlo.
#   Simulate Merton paths (diffusion + compound Poisson jumps), price a call by
#   MC with its standard error, and confirm the COS price sits within a few SE.
#   The MC simulator is the independent ground truth for the char-func pricer.
#
# Gate 4: reduces sensibly, sanity of the smile.
#   Price a strip of strikes via COS, invert to implied vol, plot. With mu_j < 0
#   the short-dated smile should show a left skew that a pure diffusion cannot.

## 4. Tuesday's plan

1. Write `merton_char_func` in `models/merton.py` (Cell 6 spec).
2. Validation gates in order (Cell 7): BSM limit, martingale check, COS-vs-MC,
   then the smile shape. Ground truth before real data, as always.
3. Once certified, the char func drops into the existing COS pricer with no
   pricer changes, that is the payoff of the Week 5 Fourier work.
4. Wednesday: Kou (double-exponential jumps). Thursday: calibrate both to the
   real 29 DTE SPX slice and test whether jumps capture the skew Heston missed.